In [1]:
import json
import sys
from pathlib import Path
from typing import Iterable
from dataclasses import dataclass

# --- CONFIGURATION ---
# Root folder containing multiple projects (each project may contain a 'simal_schemas' folder).
# In your repo, this is currently where the project folders live.
PROJECTS_ROOT = Path(r"schemas/simal")

# Preferred schemas subfolder name inside each project.
# If it doesn't exist, the code will fall back to other reasonable layouts.
SCHEMAS_SUBDIR_CANDIDATES = ("simal_schemas", "siml_schemas")

# Input file extensions treated as SIMAL schemas.
SIMAL_EXTENSIONS = {".simal", ".siml", ".txt"}

# Root output folder. Output structure will be:
#   OUT_ROOT/<project_name>/json_schemas/<schema_stem>_max_simple.json
OUT_ROOT = Path(r"schemas/max_simple_json")

# Path to the SIMAL language implementation (simal_cli.py folder).
LANGUAGE_DIR = Path(r"./")

# --- Imports from SIMAL toolchain (version-pinned via LANGUAGE_DIR) ---
if str(LANGUAGE_DIR) not in sys.path:
    sys.path.insert(0, str(LANGUAGE_DIR))

from simal_parser import parse_dsl
from simal_conversion import system_to_simple_json_dict


@dataclass(frozen=True)
class ConvertResult:
    project: str
    total: int
    converted: int
    failed: int
    failures: list[dict]


def _find_schema_dir(project_dir: Path) -> Path | None:
    """Return the directory containing schemas for a project, or None."""
    for candidate in SCHEMAS_SUBDIR_CANDIDATES:
        candidate_dir = project_dir / candidate
        if candidate_dir.is_dir():
            return candidate_dir

    # Fallback: if project_dir itself contains SIMAL files, treat it as the schema directory.
    for ext in SIMAL_EXTENSIONS:
        if any(project_dir.glob(f"*{ext}")):
            return project_dir

    return None


def _iter_project_dirs(projects_root: Path) -> Iterable[Path]:
    for child in sorted(projects_root.iterdir()):
        if child.is_dir() and not child.name.startswith("."):
            yield child


def convert_project(project_dir: Path, out_root: Path) -> ConvertResult:
    schema_dir = _find_schema_dir(project_dir)
    project_name = project_dir.name
    failures: list[dict] = []

    if schema_dir is None:
        return ConvertResult(project=project_name, total=0, converted=0, failed=0, failures=[])

    schema_files = [
        p for p in sorted(schema_dir.iterdir())
        if p.is_file() and p.suffix.lower() in SIMAL_EXTENSIONS
    ]

    out_dir = out_root / project_name / "json_schemas"
    out_dir.mkdir(parents=True, exist_ok=True)

    converted = 0
    for schema_path in schema_files:
        try:
            content = schema_path.read_text(encoding="utf-8")
            system = parse_dsl(content)
            as_dict = system_to_simple_json_dict(system, max_simplify=True)
            out_path = out_dir / f"{schema_path.stem}_max_simple.json"
            # original 4 spaces:
            #out_path.write_text(json.dumps(as_dict, indent=4, ensure_ascii=False), encoding="utf-8")
            # original 2 spaces:
            #out_path.write_text(json.dumps(as_dict, indent=2, ensure_ascii=False), encoding="utf-8")
            # no indent:
            #out_path.write_text(json.dumps(as_dict, indent=0, ensure_ascii=False), encoding="utf-8")
            # single line:
            out_path.write_text(json.dumps(as_dict, ensure_ascii=False), encoding="utf-8")

            converted += 1
        except Exception as e:
            failures.append({"file": str(schema_path), "error": str(e)})

    return ConvertResult(
        project=project_name,
        total=len(schema_files),
        converted=converted,
        failed=len(failures),
        failures=failures,
    )


def convert_all_projects(projects_root: Path, out_root: Path) -> list[ConvertResult]:
    out_root.mkdir(parents=True, exist_ok=True)
    results: list[ConvertResult] = []
    for project_dir in _iter_project_dirs(projects_root):
        results.append(convert_project(project_dir, out_root))
    return results


def print_report(results: list[ConvertResult]) -> None:
    total_projects = len(results)
    total_files = sum(r.total for r in results)
    total_converted = sum(r.converted for r in results)
    total_failed = sum(r.failed for r in results)

    print("=" * 80)
    print("MAX-SIMPLE JSON CONVERSION REPORT")
    print("=" * 80)
    print(f"Projects scanned: {total_projects}")
    print(f"Schemas found:    {total_files}")
    print(f"Converted:        {total_converted}")
    print(f"Failed:           {total_failed}")
    print(f"Output root:      {OUT_ROOT}")

    if total_failed:
        print("\nFailures:")
        for r in results:
            for f in r.failures:
                print(f"- [{r.project}] {f['file']} -> {f['error']}")

In [ ]:
# Run conversion across all projects
results = convert_all_projects(PROJECTS_ROOT, OUT_ROOT)
print_report(results)

# Optional: list projects with no schemas folder/files found
no_schema = [r.project for r in results if r.total == 0]
if no_schema:
    print("\nProjects with no schemas detected:")
    for name in no_schema:
        print("-", name)